<a href="https://colab.research.google.com/github/velchan15/MachineLearning-InternshipStarter-FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# cek koneksi jalan
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:15} {n:,} rows')

dim_clients     104 rows
dim_content     519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily      78,835,655 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Feature: gsc_impressions, gsc_clicks, gsc_avg_position — all knowable before the prediction moment, since they're historical facts.
- Label/proxy: whether impressions declined more than 20% compared to the prior period — this is computed, not a raw column.
- Context: client_hash_id, content_hash_id — used only for grouping/joining, never as a feature (they're IDs, not signal).
- Excluded: GA4 engagement columns (ga4_engaged_sessions, etc.) for rows where ga4_data_available = FALSE — these are zero-filled placeholders, not real zero engagement, so including them would inject false signal.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**A. Query**

In [10]:
#Query 1 — Check "grain"
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1,2,3
    HAVING c > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [11]:
#Query 2 — Row count & date span
con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [12]:
#Query 3 — Availability check use IS TRUE
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows
0,9841378,413966.0


**B. Fitur**

In [13]:
features = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_prev30,
        SUM(gsc_clicks) AS clicks_prev30,
        AVG(gsc_avg_position) AS avg_position_prev30,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_prev30,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_sessions_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1, 2
    HAVING impressions_prev30 >= 10
""").df()
print(f"{len(features):,} content items")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

143,206 content items


,client_hash_id,content_hash_id,impressions_prev30,clicks_prev30,avg_position_prev30,ctr_prev30,engaged_sessions_prev30
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.147402,0.000000,0.0
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,4.828125,0.021739,0.0
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.145765,0.001112,0.0
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,4.909314,0.000000,0.0
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,0.000000,0.0


In [14]:
# Leakage
# Pull NEXT month's data (April) — this represents the "future" we're trying to predict
next_month = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_next30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01' AND report_date < '2026-05-01'
    GROUP BY 1, 2
""").df()

leaky = features.merge(next_month, on=['client_hash_id', 'content_hash_id'], how='inner')
leaky['is_declining'] = (leaky['impressions_next30'] < 0.8 * leaky['impressions_prev30']).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# LEAKY VERSION — deliberately include impressions_next30 (future data!) as a "feature"
X_leak = leaky[['impressions_prev30', 'ctr_prev30', 'impressions_next30']].fillna(0)
y = leaky['is_declining']
Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.25, random_state=42)
model_leak = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("LEAKY score (using future data as a feature):", model_leak.score(Xte, yte))

# HONEST VERSION — remove the leaked column
X_honest = leaky[['impressions_prev30', 'ctr_prev30']].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.25, random_state=42)
model_honest = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("HONEST score (without leaked data):", model_honest.score(Xte, yte))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LEAKY score (using future data as a feature): 1.0
HONEST score (without leaked data): 0.5243003184179654


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can only reliably measure search-based signals (impressions, clicks, position) — GA4 engagement data is available for only ~4.2% of rows in this month (413,966 out of 9,841,378), so any engagement-based feature would be undefined for the vast majority of pages. Additionally, this single-month view can't distinguish a genuinely declining page from one still ramping up (a page with only 2 weeks of history looks identical to a 2-year-old page in this window alone) — history depth
per client would need to be checked before trusting any trend computed this way.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.